In [1]:
#use random forest

rs2909 = 2909

# 5 Modeling<a id='5_Modeling'></a>

## 5.3 Imports<a id='5.3_Imports'></a>

In [2]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import __version__ as sklearn_version
from sklearn.model_selection import cross_validate


import time
from sklearn.decomposition import PCA
from sklearn.preprocessing import scale
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
import datetime

pd.options.display.max_columns = 999
pd.options.display.max_rows = 999

## 5.4 Functions

In [3]:
#return the first 5 and last 5 rows of this dataframe
def p(df_):
    if df_.shape[0] > 6:
        print(df_.shape)
        return pd.concat([df_.head(), df_.tail()])
    else:
        return df_

def rcp(file_, pd_=None, ic_=None):
    if (pd_ == None) and (ic_ == None):
        return pd.read_csv(os.path.join('..', 'processed_data', file_))
    elif (pd_ != None) and (ic_ == None):
        return pd.read_csv(os.path.join('..', 'processed_data', file_), parse_dates=pd_)
    elif (pd_ != None) and (ic_ != None):
        return pd.read_csv(os.path.join('..', 'processed_data', file_), parse_dates=pd_, index_col = ic_)
    else:
        return pd.read_csv(os.path.join('..', 'processed_data', file_), index_col = ic_)
    
def rcr(file_, pd_=None):
    if pd_ == None:
        return pd.read_csv(os.path.join('..', 'raw_data', file_))
    else:
        return pd.read_csv(os.path.join('..', 'raw_data', file_), parse_dates=pd_)
    
#sort dataframe by column
def s(df_, column_):
    return df_.sort_values(column_)

#reset index and sort dataframe by column
def sr(df_, column_, ascending_=True):
    df_ = df_.sort_values(column_, ascending=ascending_)
    return df_.reset_index(drop=True)

#print length of list
def pl(list_):
    print(len(list_))
    return list_

#print length of dictionary
def pdc(dict_):
    print(len(dict_))
    return dict_

## 5.5 Load Data<a id='5.5_Load_Data'></a>

In [4]:
df_train = rcp('150_train_split.csv', None, 0)
p(df_train)

(7186, 88)


,E_OFF_RATING_cma_diff,OFF_RATING_cma_diff,E_DEF_RATING_cma_diff,DEF_RATING_cma_diff,E_NET_RATING_cma_diff,NET_RATING_cma_diff,AST_PCT_cma_diff,AST_TOV_cma_diff,AST_RATIO_cma_diff,OREB_PCT_cma_diff,DREB_PCT_cma_diff,REB_PCT_cma_diff,E_TM_TOV_PCT_cma_diff,TM_TOV_PCT_cma_diff,EFG_PCT_cma_diff,TS_PCT_cma_diff,USG_PCT_cma_diff,E_USG_PCT_cma_diff,E_PACE_cma_diff,PACE_cma_diff,PACE_PER40_cma_diff,POSS_cma_diff,PIE_cma_diff,duration_minutes_cma_diff,w_pct_cma_diff,min_cma_diff,fgm_cma_diff,fga_cma_diff,fg_pct_cma_diff,fg3m_cma_diff,fg3a_cma_diff,fg3_pct_cma_diff,ftm_cma_diff,fta_cma_diff,ft_pct_cma_diff,oreb_cma_diff,dreb_cma_diff,reb_cma_diff,ast_cma_diff,stl_cma_diff,blk_cma_diff,tov_cma_diff,pf_cma_diff,pts_cma_diff,spread_cma_diff,away_home_cma_diff,E_OFF_RATING_sm_cma_diff,OFF_RATING_sm_cma_diff,E_DEF_RATING_sm_cma_diff,DEF_RATING_sm_cma_diff,E_NET_RATING_sm_cma_diff,NET_RATING_sm_cma_diff,AST_PCT_sm_cma_diff,AST_TOV_sm_cma_diff,AST_RATIO_sm_cma_diff,OREB_PCT_sm_cma_diff,DREB_PCT_sm_cma_diff,REB_PCT_sm_cma_diff,E_TM_TOV_PCT_sm_cma_diff,TM_TOV_PCT_sm_cma_diff,EFG_PCT_sm_cma_diff,TS_PCT_sm_cma_diff,E_USG_PCT_sm_cma_diff,POSS_sm_cma_diff,PIE_sm_cma_diff,w_pct_sm_cma_diff,fgm_sm_cma_diff,fga_sm_cma_diff,fg_pct_sm_cma_diff,fg3m_sm_cma_diff,fg3a_sm_cma_diff,fg3_pct_sm_cma_diff,ftm_sm_cma_diff,fta_sm_cma_diff,ft_pct_sm_cma_diff,oreb_sm_cma_diff,dreb_sm_cma_diff,reb_sm_cma_diff,ast_sm_cma_diff,stl_sm_cma_diff,blk_sm_cma_diff,tov_sm_cma_diff,pf_sm_cma_diff,pts_sm_cma_diff,spread_sm_cma_diff,away_home_sm_cma_diff,away_home_a,spread_a
3134,-0.018902,-0.016231,1.321667,1.343521,-1.228430,-1.303603,-1.318706,-1.481543,-1.903444,0.921242,-2.507811,-0.529573,0.446845,0.430833,-0.575159,0.121855,-0.00658,-0.050716,0.174116,0.184420,0.184667,0.087111,-0.946293,-0.146579,-0.692156,-0.146579,-1.441489,-0.954681,-0.761156,0.058212,-0.155516,0.712864,2.741520,2.536967,1.145528,-0.129427,-0.499197,-0.454521,-1.849687,0.374410,0.695530,0.314062,1.079520,0.019169,1.242246,-1.671873,-0.972992,-0.574376,0.972992,0.574376,-0.982755,-0.574453,0.482854,-0.689375,-0.331362,0.453289,0.228158,0.772176,0.513350,0.650436,-0.657641,-0.640307,-0.502838,-1.467600,-0.218142,-0.622246,-1.140080,-0.426922,-0.790581,-0.024232,-0.034710,-0.003680,0.702370,1.160200,-1.334506,0.042893,0.400880,0.340858,-0.223699,0.020952,0.037745,-0.000575,-2.090760,-0.652068,0.652068,1.632487,-0.999416,0.819762
8383,-0.099050,-0.203595,-0.808132,-0.713716,0.644865,0.483580,1.728190,1.736950,1.631711,-1.010155,0.087533,-0.470397,-0.467579,-0.507828,0.560740,0.066701,-0.00658,-0.015186,0.085137,0.074908,0.074809,0.087111,0.662864,-0.003876,1.124071,-0.003876,0.038328,-0.153029,0.213363,1.076767,0.781105,0.678480,-1.100526,-0.236015,-1.452563,-0.918119,1.069529,0.319236,1.475443,-0.610516,0.797801,-0.417220,-1.175610,-0.119667,-0.495752,0.000029,1.384239,1.275388,-1.384239,-1.275388,1.384224,1.275392,-0.650163,-0.418961,-0.082013,-0.146715,-0.071707,0.552885,0.329894,0.273434,1.586312,1.831840,0.891767,1.413205,1.547778,2.450526,0.357308,-2.027860,1.476410,0.911499,0.124828,1.707674,1.539383,1.691886,-0.160638,-1.544978,1.529618,0.260294,-0.328340,-2.382964,2.221644,0.432133,-2.362190,1.504139,-1.504139,1.632487,-0.999416,-0.715532
7423,0.556110,0.547723,0.560475,0.683780,0.017205,-0.111224,0.252549,0.728169,0.622166,0.229666,-1.268548,-0.368039,-0.561384,-0.570175,0.448871,0.380389,-0.00658,-0.008080,-0.414064,-0.477483,-0.477456,-0.144717,0.089375,0.281530,0.373883,0.281530,0.584722,0.423159,0.227545,0.949448,0.943996,0.288787,-0.712005,-1.120647,0.844477,0.002022,0.788563,0.607145,0.626962,-1.908827,0.797801,-0.521689,-0.173330,0.418323,-0.008731,0.835980,-0.513833,-0.527205,0.513833,0.527205,-0.513886,-0.523911,-0.286039,-0.251561,-0.269025,0.644595,-1.124856,-0.262426,0.240646,0.237529,-0.547797,-0.640307,0.581855,-0.507332,-0.604055,0.553294,0.088546,1.013922,-0.565101,-0.024232,0.337545,-0.158211,-1.157660,-1.624824,0.450235,0.748613,-0.009571,0.448277,-0.223699,0.144230,0.302460,0.4

In [5]:
df_test = rcp('150_test_split.csv', None, 0)
p(df_test)

(3080, 88)


,E_OFF_RATING_cma_diff,OFF_RATING_cma_diff,E_DEF_RATING_cma_diff,DEF_RATING_cma_diff,E_NET_RATING_cma_diff,NET_RATING_cma_diff,AST_PCT_cma_diff,AST_TOV_cma_diff,AST_RATIO_cma_diff,OREB_PCT_cma_diff,DREB_PCT_cma_diff,REB_PCT_cma_diff,E_TM_TOV_PCT_cma_diff,TM_TOV_PCT_cma_diff,EFG_PCT_cma_diff,TS_PCT_cma_diff,USG_PCT_cma_diff,E_USG_PCT_cma_diff,E_PACE_cma_diff,PACE_cma_diff,PACE_PER40_cma_diff,POSS_cma_diff,PIE_cma_diff,duration_minutes_cma_diff,w_pct_cma_diff,min_cma_diff,fgm_cma_diff,fga_cma_diff,fg_pct_cma_diff,fg3m_cma_diff,fg3a_cma_diff,fg3_pct_cma_diff,ftm_cma_diff,fta_cma_diff,ft_pct_cma_diff,oreb_cma_diff,dreb_cma_diff,reb_cma_diff,ast_cma_diff,stl_cma_diff,blk_cma_diff,tov_cma_diff,pf_cma_diff,pts_cma_diff,spread_cma_diff,away_home_cma_diff,E_OFF_RATING_sm_cma_diff,OFF_RATING_sm_cma_diff,E_DEF_RATING_sm_cma_diff,DEF_RATING_sm_cma_diff,E_NET_RATING_sm_cma_diff,NET_RATING_sm_cma_diff,AST_PCT_sm_cma_diff,AST_TOV_sm_cma_diff,AST_RATIO_sm_cma_diff,OREB_PCT_sm_cma_diff,DREB_PCT_sm_cma_diff,REB_PCT_sm_cma_diff,E_TM_TOV_PCT_sm_cma_diff,TM_TOV_PCT_sm_cma_diff,EFG_PCT_sm_cma_diff,TS_PCT_sm_cma_diff,E_USG_PCT_sm_cma_diff,POSS_sm_cma_diff,PIE_sm_cma_diff,w_pct_sm_cma_diff,fgm_sm_cma_diff,fga_sm_cma_diff,fg_pct_sm_cma_diff,fg3m_sm_cma_diff,fg3a_sm_cma_diff,fg3_pct_sm_cma_diff,ftm_sm_cma_diff,fta_sm_cma_diff,ft_pct_sm_cma_diff,oreb_sm_cma_diff,dreb_sm_cma_diff,reb_sm_cma_diff,ast_sm_cma_diff,stl_sm_cma_diff,blk_sm_cma_diff,tov_sm_cma_diff,pf_sm_cma_diff,pts_sm_cma_diff,spread_sm_cma_diff,away_home_sm_cma_diff,away_home_a,spread_a
2583,0.364128,0.314684,-0.920580,-0.982429,1.188895,1.256641,-1.064798,-0.921805,-0.600467,0.252360,-0.514153,-0.408023,0.720669,0.669828,0.574508,0.578599,-0.00658,-0.064928,0.449053,0.506638,0.506363,0.416552,1.000424,-0.146579,1.079977,-0.146579,0.607488,-0.115451,0.802937,-0.387406,-0.101220,-1.221274,0.561482,0.733678,-0.036432,0.100608,-0.101162,-0.022657,-0.496703,2.433799,0.490987,0.801583,1.385772,0.583191,-1.202410,0.000029,-0.022339,-0.018436,0.022339,0.018436,-0.022383,-0.018489,-0.025225,-0.009476,-0.019676,0.018504,0.010865,0.029961,0.002373,0.004146,-0.029962,-0.034884,-0.037969,-0.027198,-0.027628,-0.034476,0.011757,0.053359,-0.022729,-0.024232,-0.034710,-0.003680,-0.041642,-0.055083,0.009755,0.042893,-0.009571,0.018601,-0.014416,0.020952,0.037745,-0.000575,0.080677,-0.020983,0.020983,0.002886,1.000585,0.527325
9370,-0.726251,-0.886865,-0.028679,-0.236006,-0.661512,-0.653382,-1.661853,-0.376687,-1.774194,-0.807101,0.385914,-0.289671,-1.054252,-1.086265,-1.014029,-1.034655,-0.00658,-0.086247,-0.983940,-0.785825,-0.785623,-0.803600,-0.960811,-0.003876,-1.871279,-0.003876,-1.487022,-0.340916,-1.206882,-0.164597,-0.060497,-0.274263,-0.668836,-0.593270,-0.465613,-0.162289,-0.147990,-0.202600,-2.468849,-1.058209,-3.139654,-1.636023,0.494856,-1.447289,0.783873,1.253955,-3.175014,-2.959863,3.175014,2.959863,-3.175121,-2.960045,-1.397040,-0.488496,-2.419660,0.435898,0.219467,-0.779727,-1.174939,-1.019147,-3.199741,-3.566522,1.821504,-0.027198,-2.562932,-1.894508,-2.291917,1.654297,-2.984443,-1.895694,-1.630092,-0.743458,-0.785654,0.552558,-2.471957,1.101473,-2.472272,-1.270426,-2.525811,1.870118,-3.138836,-1.154463,-0.733612,-3.176408,3.176408,1.632487,-0.999416,1.112200
5655,-0.544521,-0.381636,0.635441,0.584578,-1.099025,-0.938778,0.475648,0.308365,0.290309,0.386136,-1.368477,-1.277268,0.254977,0.351168,-0.613022,-0.638236,-0.00658,-0.107565,-0.011115,-0.096173,-0.095999,-0.010501,-0.780537,0.138827,-1.522132,0.138827,0.152160,0.360530,-0.181712,-1.405961,-1.214306,-0.711235,-0.215561,-0.116930,-0.225624,0.560678,-0.780163,-0.292572,0.489370,0.329640,-0.429457,0.105124,1.330090,-0.353953,0.984411,-0.835922,-0.400660,-0.736104,0.400660,0.736104,-0.400711,-0.736188,-0.858476,1.350323,-0.643048,-0.224976,-0.110820,-1.353257,-2.178713,-2.230941,-0.893020,-1.195279,-0.037969,1.413205,-0.862959,-1.656424,0.011757,1.494203,-0.772299,-0.648053,0.603442,-1.493099,-0.785654,-0.510815,-1.166473,0.572183,-2.2670

In [6]:
X_train = df_train.drop(columns = 'spread_a')
p(X_train)


(7186, 87)


,E_OFF_RATING_cma_diff,OFF_RATING_cma_diff,E_DEF_RATING_cma_diff,DEF_RATING_cma_diff,E_NET_RATING_cma_diff,NET_RATING_cma_diff,AST_PCT_cma_diff,AST_TOV_cma_diff,AST_RATIO_cma_diff,OREB_PCT_cma_diff,DREB_PCT_cma_diff,REB_PCT_cma_diff,E_TM_TOV_PCT_cma_diff,TM_TOV_PCT_cma_diff,EFG_PCT_cma_diff,TS_PCT_cma_diff,USG_PCT_cma_diff,E_USG_PCT_cma_diff,E_PACE_cma_diff,PACE_cma_diff,PACE_PER40_cma_diff,POSS_cma_diff,PIE_cma_diff,duration_minutes_cma_diff,w_pct_cma_diff,min_cma_diff,fgm_cma_diff,fga_cma_diff,fg_pct_cma_diff,fg3m_cma_diff,fg3a_cma_diff,fg3_pct_cma_diff,ftm_cma_diff,fta_cma_diff,ft_pct_cma_diff,oreb_cma_diff,dreb_cma_diff,reb_cma_diff,ast_cma_diff,stl_cma_diff,blk_cma_diff,tov_cma_diff,pf_cma_diff,pts_cma_diff,spread_cma_diff,away_home_cma_diff,E_OFF_RATING_sm_cma_diff,OFF_RATING_sm_cma_diff,E_DEF_RATING_sm_cma_diff,DEF_RATING_sm_cma_diff,E_NET_RATING_sm_cma_diff,NET_RATING_sm_cma_diff,AST_PCT_sm_cma_diff,AST_TOV_sm_cma_diff,AST_RATIO_sm_cma_diff,OREB_PCT_sm_cma_diff,DREB_PCT_sm_cma_diff,REB_PCT_sm_cma_diff,E_TM_TOV_PCT_sm_cma_diff,TM_TOV_PCT_sm_cma_diff,EFG_PCT_sm_cma_diff,TS_PCT_sm_cma_diff,E_USG_PCT_sm_cma_diff,POSS_sm_cma_diff,PIE_sm_cma_diff,w_pct_sm_cma_diff,fgm_sm_cma_diff,fga_sm_cma_diff,fg_pct_sm_cma_diff,fg3m_sm_cma_diff,fg3a_sm_cma_diff,fg3_pct_sm_cma_diff,ftm_sm_cma_diff,fta_sm_cma_diff,ft_pct_sm_cma_diff,oreb_sm_cma_diff,dreb_sm_cma_diff,reb_sm_cma_diff,ast_sm_cma_diff,stl_sm_cma_diff,blk_sm_cma_diff,tov_sm_cma_diff,pf_sm_cma_diff,pts_sm_cma_diff,spread_sm_cma_diff,away_home_sm_cma_diff,away_home_a
3134,-0.018902,-0.016231,1.321667,1.343521,-1.228430,-1.303603,-1.318706,-1.481543,-1.903444,0.921242,-2.507811,-0.529573,0.446845,0.430833,-0.575159,0.121855,-0.00658,-0.050716,0.174116,0.184420,0.184667,0.087111,-0.946293,-0.146579,-0.692156,-0.146579,-1.441489,-0.954681,-0.761156,0.058212,-0.155516,0.712864,2.741520,2.536967,1.145528,-0.129427,-0.499197,-0.454521,-1.849687,0.374410,0.695530,0.314062,1.079520,0.019169,1.242246,-1.671873,-0.972992,-0.574376,0.972992,0.574376,-0.982755,-0.574453,0.482854,-0.689375,-0.331362,0.453289,0.228158,0.772176,0.513350,0.650436,-0.657641,-0.640307,-0.502838,-1.467600,-0.218142,-0.622246,-1.140080,-0.426922,-0.790581,-0.024232,-0.034710,-0.003680,0.702370,1.160200,-1.334506,0.042893,0.400880,0.340858,-0.223699,0.020952,0.037745,-0.000575,-2.090760,-0.652068,0.652068,1.632487,-0.999416
8383,-0.099050,-0.203595,-0.808132,-0.713716,0.644865,0.483580,1.728190,1.736950,1.631711,-1.010155,0.087533,-0.470397,-0.467579,-0.507828,0.560740,0.066701,-0.00658,-0.015186,0.085137,0.074908,0.074809,0.087111,0.662864,-0.003876,1.124071,-0.003876,0.038328,-0.153029,0.213363,1.076767,0.781105,0.678480,-1.100526,-0.236015,-1.452563,-0.918119,1.069529,0.319236,1.475443,-0.610516,0.797801,-0.417220,-1.175610,-0.119667,-0.495752,0.000029,1.384239,1.275388,-1.384239,-1.275388,1.384224,1.275392,-0.650163,-0.418961,-0.082013,-0.146715,-0.071707,0.552885,0.329894,0.273434,1.586312,1.831840,0.891767,1.413205,1.547778,2.450526,0.357308,-2.027860,1.476410,0.911499,0.124828,1.707674,1.539383,1.691886,-0.160638,-1.544978,1.529618,0.260294,-0.328340,-2.382964,2.221644,0.432133,-2.362190,1.504139,-1.504139,1.632487,-0.999416
7423,0.556110,0.547723,0.560475,0.683780,0.017205,-0.111224,0.252549,0.728169,0.622166,0.229666,-1.268548,-0.368039,-0.561384,-0.570175,0.448871,0.380389,-0.00658,-0.008080,-0.414064,-0.477483,-0.477456,-0.144717,0.089375,0.281530,0.373883,0.281530,0.584722,0.423159,0.227545,0.949448,0.943996,0.288787,-0.712005,-1.120647,0.844477,0.002022,0.788563,0.607145,0.626962,-1.908827,0.797801,-0.521689,-0.173330,0.418323,-0.008731,0.835980,-0.513833,-0.527205,0.513833,0.527205,-0.513886,-0.523911,-0.286039,-0.251561,-0.269025,0.644595,-1.124856,-0.262426,0.240646,0.237529,-0.547797,-0.640307,0.581855,-0.507332,-0.604055,0.553294,0.088546,1.013922,-0.565101,-0.024232,0.337545,-0.158211,-1.157660,-1.624824,0.450235,0.748613,-0.009571,0.448277,-0.223699,0.144230,0.302460,0.480211,1.890209,-0.581947,0.5

In [7]:
y_train = df_train.spread_a
p(y_train)

(7186,)


3134     0.819762
8383    -0.715532
7423     1.623964
2528     0.746653
10064    0.307998
3378    -0.349986
1965     1.039090
1161    -0.569314
2771    -0.276877
9937    -0.569314
Name: spread_a, dtype: float64

In [8]:
X_test = df_test.drop(columns = 'spread_a')
p(X_test)


(3080, 87)


,E_OFF_RATING_cma_diff,OFF_RATING_cma_diff,E_DEF_RATING_cma_diff,DEF_RATING_cma_diff,E_NET_RATING_cma_diff,NET_RATING_cma_diff,AST_PCT_cma_diff,AST_TOV_cma_diff,AST_RATIO_cma_diff,OREB_PCT_cma_diff,DREB_PCT_cma_diff,REB_PCT_cma_diff,E_TM_TOV_PCT_cma_diff,TM_TOV_PCT_cma_diff,EFG_PCT_cma_diff,TS_PCT_cma_diff,USG_PCT_cma_diff,E_USG_PCT_cma_diff,E_PACE_cma_diff,PACE_cma_diff,PACE_PER40_cma_diff,POSS_cma_diff,PIE_cma_diff,duration_minutes_cma_diff,w_pct_cma_diff,min_cma_diff,fgm_cma_diff,fga_cma_diff,fg_pct_cma_diff,fg3m_cma_diff,fg3a_cma_diff,fg3_pct_cma_diff,ftm_cma_diff,fta_cma_diff,ft_pct_cma_diff,oreb_cma_diff,dreb_cma_diff,reb_cma_diff,ast_cma_diff,stl_cma_diff,blk_cma_diff,tov_cma_diff,pf_cma_diff,pts_cma_diff,spread_cma_diff,away_home_cma_diff,E_OFF_RATING_sm_cma_diff,OFF_RATING_sm_cma_diff,E_DEF_RATING_sm_cma_diff,DEF_RATING_sm_cma_diff,E_NET_RATING_sm_cma_diff,NET_RATING_sm_cma_diff,AST_PCT_sm_cma_diff,AST_TOV_sm_cma_diff,AST_RATIO_sm_cma_diff,OREB_PCT_sm_cma_diff,DREB_PCT_sm_cma_diff,REB_PCT_sm_cma_diff,E_TM_TOV_PCT_sm_cma_diff,TM_TOV_PCT_sm_cma_diff,EFG_PCT_sm_cma_diff,TS_PCT_sm_cma_diff,E_USG_PCT_sm_cma_diff,POSS_sm_cma_diff,PIE_sm_cma_diff,w_pct_sm_cma_diff,fgm_sm_cma_diff,fga_sm_cma_diff,fg_pct_sm_cma_diff,fg3m_sm_cma_diff,fg3a_sm_cma_diff,fg3_pct_sm_cma_diff,ftm_sm_cma_diff,fta_sm_cma_diff,ft_pct_sm_cma_diff,oreb_sm_cma_diff,dreb_sm_cma_diff,reb_sm_cma_diff,ast_sm_cma_diff,stl_sm_cma_diff,blk_sm_cma_diff,tov_sm_cma_diff,pf_sm_cma_diff,pts_sm_cma_diff,spread_sm_cma_diff,away_home_sm_cma_diff,away_home_a
2583,0.364128,0.314684,-0.920580,-0.982429,1.188895,1.256641,-1.064798,-0.921805,-0.600467,0.252360,-0.514153,-0.408023,0.720669,0.669828,0.574508,0.578599,-0.00658,-0.064928,0.449053,0.506638,0.506363,0.416552,1.000424,-0.146579,1.079977,-0.146579,0.607488,-0.115451,0.802937,-0.387406,-0.101220,-1.221274,0.561482,0.733678,-0.036432,0.100608,-0.101162,-0.022657,-0.496703,2.433799,0.490987,0.801583,1.385772,0.583191,-1.202410,0.000029,-0.022339,-0.018436,0.022339,0.018436,-0.022383,-0.018489,-0.025225,-0.009476,-0.019676,0.018504,0.010865,0.029961,0.002373,0.004146,-0.029962,-0.034884,-0.037969,-0.027198,-0.027628,-0.034476,0.011757,0.053359,-0.022729,-0.024232,-0.034710,-0.003680,-0.041642,-0.055083,0.009755,0.042893,-0.009571,0.018601,-0.014416,0.020952,0.037745,-0.000575,0.080677,-0.020983,0.020983,0.002886,1.000585
9370,-0.726251,-0.886865,-0.028679,-0.236006,-0.661512,-0.653382,-1.661853,-0.376687,-1.774194,-0.807101,0.385914,-0.289671,-1.054252,-1.086265,-1.014029,-1.034655,-0.00658,-0.086247,-0.983940,-0.785825,-0.785623,-0.803600,-0.960811,-0.003876,-1.871279,-0.003876,-1.487022,-0.340916,-1.206882,-0.164597,-0.060497,-0.274263,-0.668836,-0.593270,-0.465613,-0.162289,-0.147990,-0.202600,-2.468849,-1.058209,-3.139654,-1.636023,0.494856,-1.447289,0.783873,1.253955,-3.175014,-2.959863,3.175014,2.959863,-3.175121,-2.960045,-1.397040,-0.488496,-2.419660,0.435898,0.219467,-0.779727,-1.174939,-1.019147,-3.199741,-3.566522,1.821504,-0.027198,-2.562932,-1.894508,-2.291917,1.654297,-2.984443,-1.895694,-1.630092,-0.743458,-0.785654,0.552558,-2.471957,1.101473,-2.472272,-1.270426,-2.525811,1.870118,-3.138836,-1.154463,-0.733612,-3.176408,3.176408,1.632487,-0.999416
5655,-0.544521,-0.381636,0.635441,0.584578,-1.099025,-0.938778,0.475648,0.308365,0.290309,0.386136,-1.368477,-1.277268,0.254977,0.351168,-0.613022,-0.638236,-0.00658,-0.107565,-0.011115,-0.096173,-0.095999,-0.010501,-0.780537,0.138827,-1.522132,0.138827,0.152160,0.360530,-0.181712,-1.405961,-1.214306,-0.711235,-0.215561,-0.116930,-0.225624,0.560678,-0.780163,-0.292572,0.489370,0.329640,-0.429457,0.105124,1.330090,-0.353953,0.984411,-0.835922,-0.400660,-0.736104,0.400660,0.736104,-0.400711,-0.736188,-0.858476,1.350323,-0.643048,-0.224976,-0.110820,-1.353257,-2.178713,-2.230941,-0.893020,-1.195279,-0.037969,1.413205,-0.862959,-1.656424,0.011757,1.494203,-0.772299,-0.648053,0.603442,-1.493099,-0.785654,-0.510815,-1.166473,0.572183,-2.267047,-1.431554,-0.642265,1.87

In [9]:
y_test = df_test.spread_a
p(y_test)


(3080,)


2583    0.527325
9370    1.112200
5655   -0.057549
7258   -0.349986
2386   -0.715532
8109    0.161779
9743    1.770183
4546    0.454216
4962    0.746653
5706    0.381107
Name: spread_a, dtype: float64

## 5.6 RF Model

In [10]:

#Code task 22#
#Define a pipeline comprising the steps:
#SimpleImputer() with a strategy of 'median'
#StandardScaler(),
#and then RandomForestRegressor() with a random state of 47
RF_pipe = make_pipeline(
    #StandardScaler(),
    RandomForestRegressor(random_state=rs2909)
)

In [11]:
rf_default_cv_results = cross_validate(RF_pipe, X_train, y_train, cv=5)


In [12]:
rf_cv_scores = rf_default_cv_results['test_score']
rf_cv_scores

array([0.14191887, 0.15810184, 0.15355164, 0.14914375, 0.13823307])

In [13]:
np.mean(rf_cv_scores), np.std(rf_cv_scores)

(0.14818983635764027, 0.007298887983183603)

In [14]:
start_time = time.time()

In [15]:
n_est = [int(n) for n in np.logspace(start=1, stop=3, num=20)]
grid_params = {
        'randomforestregressor__n_estimators': n_est,
        #'standardscaler': [StandardScaler(), None],
        #'simpleimputer__strategy': ['mean', 'median']
}
grid_params


{'randomforestregressor__n_estimators': [10,
  12,
  16,
  20,
  26,
  33,
  42,
  54,
  69,
  88,
  112,
  143,
  183,
  233,
  297,
  379,
  483,
  615,
  784,
  1000]}

In [16]:
print("--- %s seconds ---" % (time.time() - start_time))

--- 0.0329129695892334 seconds ---


In [17]:
start_time = time.time()

In [18]:
rf_grid_cv = GridSearchCV(RF_pipe, param_grid=grid_params, cv=5, n_jobs=-1)

In [19]:
print("--- %s seconds ---" % (time.time() - start_time))

--- 0.03391075134277344 seconds ---


In [20]:
start_time = time.time()

In [21]:
rf_grid_cv.fit(X_train, y_train)

GridSearchCV(cv=5, error_score=nan,
             estimator=Pipeline(memory=None,
                                steps=[('randomforestregressor',
                                        RandomForestRegressor(bootstrap=True,
                                                              ccp_alpha=0.0,
                                                              criterion='mse',
                                                              max_depth=None,
                                                              max_features='auto',
                                                              max_leaf_nodes=None,
                                                              max_samples=None,
                                                              min_impurity_decrease=0.0,
                                                              min_impurity_split=None,
                                                              min_samples_leaf=1,
                                      

In [22]:
print("--- %s seconds ---" % (time.time() - start_time))

--- 1416.2506625652313 seconds ---


In [31]:
1416 / 60

23.6

In [23]:
rf_grid_cv.best_params_

{'randomforestregressor__n_estimators': 1000}

In [24]:
start_time = time.time()

In [25]:
rf_best_cv_results = cross_validate(rf_grid_cv.best_estimator_, X_train, y_train, cv=5)
rf_best_scores = rf_best_cv_results['test_score']
rf_best_scores

array([0.15138007, 0.16280553, 0.16487388, 0.16750328, 0.1469421 ])

<font color='red'> ?????????</font>

In [26]:
print("--- %s seconds ---" % (time.time() - start_time))

--- 1182.3001058101654 seconds ---


In [27]:
np.mean(rf_best_scores), np.std(rf_best_scores)

(0.15870097316767975, 0.008053563783215421)

In [28]:
start_time = time.time()

In [1]:
plt.subplots(figsize=(10, 5))
imps = rf_grid_cv.best_estimator_.named_steps.randomforestregressor.feature_importances_
rf_feat_imps = pd.Series(imps, index=X_train.columns).sort_values(ascending=False)
rf_feat_imps.plot(kind='bar')
plt.xlabel('features')
plt.ylabel('importance')
plt.title('Best random forest regressor feature importances');

NameError: name 'plt' is not defined

In [30]:
print("--- %s seconds ---" % (time.time() - start_time))

--- 1.5200610160827637 seconds ---


<font color='red'> add max of stat features?????</font>

This next step requires some careful thought. We want to refit the model using all available data. But should we include Big Mountain data? On the one hand, we are _not_ trying to estimate model performance on a previously unseen data sample, so theoretically including Big Mountain data should be fine. One might first think that including Big Mountain in the model training would, if anything, improve model performance in predicting Big Mountain's ticket price. But here's where our business context comes in. The motivation for this entire project is based on the sense that Big Mountain needs to adjust its pricing. One way to phrase this problem: we want to train a model to predict Big Mountain's ticket price based on data from _all the other_ resorts! We don't want Big Mountain's current price to bias this. We want to calculate a price based only on its competitors.

In [ ]:
X = ski_data.loc[ski_data.Name != "Big Mountain Resort", model.X_columns]
y = ski_data.loc[ski_data.Name != "Big Mountain Resort", 'AdultWeekend']

In [ ]:
len(X), len(y)

In [ ]:
model.fit(X, y)

In [ ]:
cv_results = cross_validate(model, X, y, scoring='neg_mean_absolute_error', cv=5, n_jobs=-1)

In [ ]:
cv_results['test_score']

<font color='red'> why is there is such large numbers in this test_score? what does it mean? I thought we'd get smaller numbers like in RF_pipe of '04_preprocessing_and_training'<font>

In [ ]:
mae_mean, mae_std = np.mean(-1 * cv_results['test_score']), np.std(-1 * cv_results['test_score'])
mae_mean, mae_std

These numbers will inevitably be different to those in the previous step that used a different training data set. They should, however, be consistent. It's important to appreciate that estimates of model performance are subject to the noise and uncertainty of data!

## 5.7 Calculate Expected Big Mountain Ticket Price From The Model<a id='5.7_Calculate_Expected_Big_Mountain_Ticket_Price_From_The_Model'></a>

In [ ]:
X_bm = ski_data.loc[ski_data.Name == "Big Mountain Resort", model.X_columns]
y_bm = ski_data.loc[ski_data.Name == "Big Mountain Resort", 'AdultWeekend']

In [ ]:
bm_pred = model.predict(X_bm).item()

In [ ]:
y_bm = y_bm.values.item()

In [ ]:
print(f'Big Mountain Resort modelled price is ${bm_pred:.2f}, actual price is ${y_bm:.2f}.')
print(f'Even with the expected mean absolute error of ${mae_mean:.2f}, this suggests there is room for an increase.')

This result should be looked at optimistically and doubtfully! The validity of our model lies in the assumption that other resorts accurately set their prices according to what the market (the ticket-buying public) supports. The fact that our resort seems to be charging that much less that what's predicted suggests our resort might be undercharging. 
But if ours is mispricing itself, are others? It's reasonable to expect that some resorts will be "overpriced" and some "underpriced." Or if resorts are pretty good at pricing strategies, it could be that our model is simply lacking some key data? Certainly we know nothing about operating costs, for example, and they would surely help.

## 5.8 Big Mountain Resort In Market Context<a id='5.8_Big_Mountain_Resort_In_Market_Context'></a>

Features that came up as important in the modeling (not just our final, random forest model) included:
* vertical_drop
* Snow Making_ac
* total_chairs
* fastQuads
* Runs
* LongestRun_mi
* trams
* SkiableTerrain_ac

A handy glossary of skiing terms can be found on the [ski.com](https://www.ski.com/ski-glossary) site. Some potentially relevant contextual information is that vertical drop, although nominally the height difference from the summit to the base, is generally taken from the highest [_lift-served_](http://verticalfeet.com/) point.

It's often useful to define custom functions for visualizing data in meaningful ways. The function below takes a feature name as an input and plots a histogram of the values of that feature. It then marks where Big Mountain sits in the distribution by marking Big Mountain's value with a vertical line using `matplotlib`'s [axvline](https://matplotlib.org/3.1.1/api/_as_gen/matplotlib.pyplot.axvline.html) function. It also performs a little cleaning up of missing values and adds descriptive labels and a title.

In [ ]:
#Code task 1#
#Add code to the `plot_compare` function that displays a vertical, dashed line
#on the histogram to indicate Big Mountain's position in the distribution
#Hint: plt.axvline() plots a vertical line, its position for 'feature1'
#would be `big_mountain['feature1'].values, we'd like a red line, which can be
#specified with c='r', a dashed linestyle is produced by ls='--',
#and it's nice to give it a slightly reduced alpha value, such as 0.8.
#Don't forget to give it a useful label (e.g. 'Big Mountain') so it's listed
#in the legend.
def plot_compare(feat_name, description, state=None, figsize=(10, 5)):
    """Graphically compare distributions of features.
    
    Plot histogram of values for all resorts and reference line to mark
    Big Mountain's position.
    
    Arguments:
    feat_name - the feature column name in the data
    description - text description of the feature
    state - select a specific state (None for all states)
    figsize - (optional) figure size
    """
    
    plt.subplots(figsize=figsize)
    # quirk that hist sometimes objects to NaNs, sometimes doesn't
    # filtering only for finite values tidies this up
    if state is None:
        ski_x = ski_data[feat_name]
    else:
        ski_x = ski_data.loc[ski_data.state == state, feat_name]
    ski_x = ski_x[np.isfinite(ski_x)]
    plt.hist(ski_x, bins=30)
    plt.axvline(x=big_mountain[feat_name].values, c='r', ls='--', alpha=0.8, label='Big Mountain')
    plt.xlabel(description)
    plt.ylabel('frequency')
    plt.title(description + ' distribution for resorts in market share')
    plt.legend()

### 5.8.1 Ticket price<a id='5.8.1_Ticket_price'></a>

Look at where Big Mountain sits overall amongst all resorts for price and for just other resorts in Montana.

In [ ]:
plot_compare('AdultWeekend', 'Adult weekend ticket price ($)')

In [ ]:
plot_compare('AdultWeekend', 'Adult weekend ticket price ($) - Montana only', state='Montana')

### 5.8.2 Vertical drop<a id='5.8.2_Vertical_drop'></a>

In [ ]:
plot_compare('vertical_drop', 'Vertical drop (feet)')

Big Mountain is doing well for vertical drop, but there are still quite a few resorts with a greater drop.

<font color='red'>Big Mountain percentile for vertical drop???????????????<font>

In [ ]:
1 - ski_data[ski_data['vertical_drop'] >= 2353][['Name','vertical_drop']].count() / ski_data['vertical_drop'].count()
#ski_data[['Name','vertical_drop']]

### 5.8.3 Snow making area<a id='5.8.3_Snow_making_area'></a>

In [ ]:
plot_compare('Snow Making_ac', 'Area covered by snow makers (acres)')

Big Mountain is very high up the league table of snow making area.

### 5.8.4 Total number of chairs<a id='5.8.4_Total_number_of_chairs'></a>

In [ ]:
plot_compare('total_chairs', 'Total number of chairs')

Big Mountain has amongst the highest number of total chairs, resorts with more appear to be outliers.

<font color='red'> how were total number of chairs counted????? May the counting method be reason for some of these outliers??????????????????<font> 

In [ ]:
ski_data[ski_data['total_chairs'] > 35 ]['AdultWeekend']

In [ ]:
ski_data[ski_data['total_chairs'] > 35 ]

### 5.8.5 Fast quads<a id='5.8.5_Fast_quads'></a>

In [ ]:
plot_compare('fastQuads', 'Number of fast quads')

Most resorts have no fast quads. Big Mountain has 3, which puts it high up that league table. There are some values  much higher, but they are rare.

<font color='red'>league table??????????????<font>

### 5.8.6 Runs<a id='5.8.6_Runs'></a>

In [ ]:
plot_compare('Runs', 'Total number of runs')

Big Mountain compares well for the number of runs. There are some resorts with more, but not many.

### 5.8.7 Longest run<a id='5.8.7_Longest_run'></a>

In [ ]:
plot_compare('LongestRun_mi', 'Longest run length (miles)')

Big Mountain has one of the longest runs. Although it is just over half the length of the longest, the longer ones are rare.

### 5.8.8 Trams<a id='5.8.8_Trams'></a>

In [ ]:
plot_compare('trams', 'Number of trams')

The vast majority of resorts, such as Big Mountain, have no trams.

### 5.8.9 Skiable terrain area<a id='5.8.9_Skiable_terrain_area'></a>

In [ ]:
plot_compare('SkiableTerrain_ac', 'Skiable terrain area (acres)')

Big Mountain is amongst the resorts with the largest amount of skiable terrain.

<font color='red'>percentile????????????<font>

In [ ]:
X_bm['SkiableTerrain_ac']

In [ ]:
1 - ski_data[ski_data['SkiableTerrain_ac'] >= 3000][['Name','SkiableTerrain_ac']].count() / ski_data['SkiableTerrain_ac'].count()

## 5.9 Modeling scenarios<a id='5.9_Modeling_scenarios'></a>

Big Mountain Resort has been reviewing potential scenarios for either cutting costs or increasing revenue (from ticket prices). Ticket price is not determined by any set of parameters; the resort is free to set whatever price it likes. However, the resort operates within a market where people pay more for certain facilities, and less for others. Being able to sense how facilities support a given ticket price is valuable business intelligence. This is where the utility of our model comes in.

The business has shortlisted some options:
1. Permanently closing down up to 10 of the least used runs. This doesn't impact any other resort statistics.
2. Increase the vertical drop by adding a run to a point 150 feet lower down but requiring the installation of an additional chair lift to bring skiers back up, without additional snow making coverage
3. Same as number 2, but adding 2 acres of snow making cover
4. Increase the longest run by 0.2 mile to boast 3.5 miles length, requiring an additional snow making coverage of 4 acres

The expected number of visitors over the season is 350,000 and, on average, visitors ski for five days. Assume the provided data includes the additional lift that Big Mountain recently installed.

<font color='red'>facilities support a given ticket price?????
    
    
how do we know 1?????????
    
    
    
    
<font>

In [ ]:
expected_visitors = 350_000

In [ ]:
all_feats = ['vertical_drop', 'Snow Making_ac', 'total_chairs', 'fastQuads', 
             'Runs', 'LongestRun_mi', 'trams', 'SkiableTerrain_ac']
big_mountain[all_feats]

In [ ]:
#Code task 2#
#In this function, copy the Big Mountain data into a new data frame
#(Note we use .copy()!)
#And then for each feature, and each of its deltas (changes from the original),
#create the modified scenario dataframe (bm2) and make a ticket price prediction
#for it. The difference between the scenario's prediction and the current
#prediction is then calculated and returned.
#Complete the code to increment each feature by the associated delta
def predict_increase(features, deltas):
    """Increase in modelled ticket price by applying delta to feature.
    
    Arguments:
    features - list, names of the features in the ski_data dataframe to change
    deltas - list, the amounts by which to increase the values of the features
    
    Outputs:
    Amount of increase in the predicted ticket price
    """
    
    bm2 = X_bm.copy()
    for f, d in zip(features, deltas):
        bm2[f] += d
    return model.predict(bm2).item() - model.predict(X_bm).item()

### 5.9.1 Scenario 1<a id='5.9.1_Scenario_1'></a>

Close up to 10 of the least used runs. The number of runs is the only parameter varying.

In [ ]:
[i for i in range(-1, -11, -1)]

In [ ]:
runs_delta = [i for i in range(-1, -11, -1)]
price_deltas = [predict_increase(['Runs'], [delta]) for delta in runs_delta]

In [ ]:
price_deltas

In [ ]:
#Code task 3#
#Create two plots, side by side, for the predicted ticket price change (delta) for each
#condition (number of runs closed) in the scenario and the associated predicted revenue
#change on the assumption that each of the expected visitors buys 5 tickets
#There are two things to do here:
#1 - use a list comprehension to create a list of the number of runs closed from `runs_delta`
#2 - use a list comprehension to create a list of predicted revenue changes from `price_deltas`
runs_closed = [-1 * rc for rc in runs_delta] #1
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
fig.subplots_adjust(wspace=0.5)
ax[0].plot(runs_closed, price_deltas, 'o-')
ax[0].set(xlabel='Runs closed', ylabel='Change ($)', title='Ticket price')
revenue_deltas = [5 * expected_visitors * pd for pd in price_deltas] #2
ax[1].plot(runs_closed, revenue_deltas, 'o-')
ax[1].set(xlabel='Runs closed', ylabel='Change ($)', title='Revenue');

The model says closing one run makes no difference. Closing 2 and 3 successively reduces support for ticket price and so revenue. If Big Mountain closes down 3 runs, it seems they may as well close down 4 or 5 as there's no further loss in ticket price. Increasing the closures down to 6 or more leads to a large drop. 

### 5.9.2 Scenario 2<a id='5.9.2_Scenario_2'></a>

In this scenario, Big Mountain is adding a run, increasing the vertical drop by 150 feet, and installing an additional chair lift.

In [ ]:
#Code task 4#
#Call `predict_increase` with a list of the features 'Runs', 'vertical_drop', and 'total_chairs'
#and associated deltas of 1, 150, and 1
ticket2_increase = predict_increase(['Runs', 'vertical_drop', 'total_chairs'], [1, 150, 1])
revenue2_increase = 5 * expected_visitors * ticket2_increase

In [ ]:
print(f'This scenario increases support for ticket price by ${ticket2_increase:.2f}')
print(f'Over the season, this could be expected to amount to ${revenue2_increase:.0f}')

<font color='red'> not getting the $1.99 and ~3.5 mil as preshown output suggested??????????? different model used????<font>

### 5.9.3 Scenario 3<a id='5.9.3_Scenario_3'></a>

In this scenario, you are repeating the previous one but adding 2 acres of snow making.

In [ ]:
#Code task 5#
#Repeat scenario 2 conditions, but add an increase of 2 to `Snow Making_ac`
ticket3_increase = predict_increase(['Runs', 'vertical_drop', 'total_chairs', 'Snow Making_ac'], [1, 150, 1, 2])
revenue3_increase = 5 * expected_visitors * ticket3_increase

In [ ]:
print(f'This scenario increases support for ticket price by ${ticket3_increase:.2f}')
print(f'Over the season, this could be expected to amount to ${revenue3_increase:.0f}')

Such a small increase in the snow making area makes no difference!

In [ ]:
print(f'percentage profit: {(revenue3_increase - 1500000) / 1500000 * 100}')

### 5.9.4 Scenario 4<a id='5.9.4_Scenario_4'></a>

This scenario calls for increasing the longest run by .2 miles and guaranteeing its snow coverage by adding 4 acres of snow making capability.

In [ ]:
#Code task 6#
#Predict the increase from adding 0.2 miles to `LongestRun_mi` and 4 to `Snow Making_ac`
predict_increase(['LongestRun_mi', 'Snow Making_ac'], [.2, 4])

No difference whatsoever. Although the longest run feature was used in the linear model, the random forest model (the one we chose because of its better performance) only has longest run way down in the feature importance list. 

## 5.10 Summary<a id='5.10_Summary'></a>

**Q: 1** Write a summary of the results of modeling these scenarios. Start by starting the current position; how much does Big Mountain currently charge? What does your modelling suggest for a ticket price that could be supported in the marketplace by Big Mountain's facilities? How would you approach suggesting such a change to the business leadership? Discuss the additional operating cost of the new chair lift per ticket (on the basis of each visitor on average buying 5 day tickets) in the context of raising prices to cover this. For future improvements, state which, if any, of the modeled scenarios you'd recommend for further consideration. Suggest how the business might test, and progress, with any run closures.

**Q: 1** Write a summary of the results of modeling these scenarios. Start by starting the current position; how much does Big Mountain currently charge? What does your modelling suggest for a ticket price that could be supported in the marketplace by Big Mountain's facilities? How would you approach suggesting such a change to the business leadership? Discuss the additional operating cost of the new chair lift per ticket (on the basis of each visitor on average buying 5 day tickets) in the context of raising prices to cover this. For future improvements, state which, if any, of the modeled scenarios you'd recommend for further consideration. Suggest how the business might test, and progress, with any run closures.

**A: 1** Your answer here

Start by starting the current position; how much does Big Mountain currently charge?
<br>-Big Mountain Resort currently charges an 'AdultWeekend' ticket price of $81.00. 

What does your modelling suggest for a ticket price that could be supported in the marketplace by Big Mountain's facilities?
<br>-The modelling suggests 'AdultWeekend' ticket price could be $96.75.

How would you approach suggesting such a change to the business leadership? Discuss the additional operating cost of the new chair lift per ticket (on the basis of each visitor on average buying 5 day tickets) in the context of raising prices to cover this.
<br>-I would approach suggesting a change to the business leadership by speaking about what the data has presented as possible solutions.
<br>-The additional operating cost of adding a chairlift to Big Mountain Resort can be offset by raising the average ticket price for each visitor. -In our ?analysis?, we assume 350,000 visitor per season, each buying an average of 5 day tickets at 81 dollars and therefore a 1.00 dollar increase would yield a 1.75 million dollar income increase.

<br>-Our model shows it would be market competitive (national) to raise the ticket price to by an average of 1.10 dollar. This would yield an annual income increase of 1,927,536 dollars and cover the additional $1,540,000 operating cost for this seasonal.

<br>-Because of Big Mountain's high rank among ticket price predictive categories (i.e. amount of facilities offered), we believe this $1.10 average increase in ticket price is justified given the trend for ticket prices nationally.

For future improvements, state which, if any, of the modeled scenarios you'd recommend for further consideration.
<br>-For future improvements, I would suggest to look more into the scenario of adding another lift while increasing the 'vertical_drop' and increasing the number of 'Runs' at a steady rate.

Suggest how the business might test, and progress, with any run closures.
<br>-Our modeling suggests a small shutdown in number of runs, e.g. 1, should not mean a decrease ticket prices, while the closer of more than that will in stages. We recommend to not to close more than 1 if we are to maximize profits or minimize shutdowns to 3-5 or less.


## 5.11 Further work<a id='5.11_Further_work'></a>

**Q: 2** What next? Highlight any deficiencies in the data that hampered or limited this work. The only price data in our dataset were ticket prices. You were provided with information about the additional operating cost of the new chair lift, but what other cost information would be useful? Big Mountain was already fairly high on some of the league charts of facilities offered, but why was its modeled price so much higher than its current price? Would this mismatch come as a surprise to the business executives? How would you find out? Assuming the business leaders felt this model was useful, how would the business make use of it? Would you expect them to come to you every time they wanted to test a new combination of parameters in a scenario? We hope you would have better things to do, so how might this model be made available for business analysts to use and explore?

**A: 2** Your answer here

**Q: 2** What next? Highlight any deficiencies in the data that hampered or limited this work. The only price data in our dataset were ticket prices. You were provided with information about the additional operating cost of the new chair lift, but what other cost information would be useful? Big Mountain was already fairly high on some of the league charts of facilities offered, but why was its modeled price so much higher than its current price? Would this mismatch come as a surprise to the business executives? How would you find out? Assuming the business leaders felt this model was useful, how would the business make use of it? Would you expect them to come to you every time they wanted to test a new combination of parameters in a scenario? We hope you would have better things to do, so how might this model be made available for business analysts to use and explore?

What next? Highlight any deficiencies in the data that hampered or limited this work. The only price data in our dataset were ticket prices.
<br>-One of the deficiencies in the data included knowing we have 350,000 visitors per season, but not knowing the precise behavior of visitor purchaser behavior like number of day tickets purchased and therefore making an assumption (of 5) on the average number of day tickets purchased per season per vistor.v

You were provided with information about the additional operating cost of the new chair lift, but what other cost information would be useful?
<br>-It would be useful to have information about the location of the this new chair lift in addition to its operating cost. Information of interest about the chair lift per se include speed of chair lift and number of chairs of the chair lift. Other information of interest include elevation spanned by the chairlift and number of runs the chair lift would add to Big Mountain Resort.
<br>-Also, it would be useful to have information about the number of tickets each person purchased each season.


Big Mountain was already fairly high on some of the league charts of facilities offered, but why was its modeled price so much higher than its current price?
<br>-Big Mountain's model price was higher than it's current price maybe because the state of Montana charges less for vistor day ticket.

Would this mismatch come as a surprise to the business executives? 
<br>-Maybe

How would you find out?
<br>-by asking

Assuming the business leaders felt this model was useful, how would the business make use of it? 
<br>-They could look into increasing vertical_drop, Snow Making_ac, total_chairs, fastQuads, Runs, LongestRun_mi, trams, and/or SkiableTerrain_ac and see if it would be the desireable and profitable to take on that scenario using our model.

Would you expect them to come to you every time they wanted to test a new combination of parameters in a scenario?
<br>-Maybe

We hope you would have better things to do, so how might this model be made available for business analysts to use and explore?
<br>-This model can be used and shared via github or other software for business analysts to use and explored.